In [1]:
from torch.utils.data import Dataset
import torch
import os
import glob
import cv2
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split, TensorDataset, Subset, ConcatDataset
from tqdm import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split

In [2]:
class CustomDatasetECPICK(Dataset):
    def __init__(self, train_df):
        self.train_df = train_df
        self.labels = {}
        self.class_labels = []
        self.classes = None
        self.data = None
        self.seqencoder = SeqEncoder()

        all_classes = []
        with open(f'utils/output_classes.pkl', 'rb') as f:
            self.classes = pickle.load(f)
        for class_group in self.classes:
            all_classes.extend(class_group)
        for i, cls in enumerate(all_classes):
            self.labels[cls] = i

    def expand_ec(self, ec):
        parts = ec.split('.')
        expanded = []
        for i in range(1, len(parts)+1):
            expanded.append('.'.join(parts[:i]))
        return expanded

    def __getitem__(self, item):
        sequence = self.train_df.iloc[item]['Sequence']
        EC = str(self.train_df.iloc[item]['EC_List']).split(',')
        x_value = []

        if len(sequence) > 1000:
            sequence = sequence[:1000]
        sequence = sequence + (' ' * (1000 - len(sequence)))
        sequence = self.seqencoder.seq_to_one_hot_encoding(sequence)
        x_value = np.reshape(np.array(sequence), (1, 1000, 21))
        
        num_classes = len(self.labels)
        one_hot_label = np.zeros(num_classes)

        for label in EC:
            if label not in self.labels:
                continue
            expanded_ecs = self.expand_ec(label)
            for e in expanded_ecs:
                if e not in self.labels:
                    continue
                label_number = self.labels[e]
                one_hot_label[label_number] = 1
        return x_value, one_hot_label
    
    def __len__(self):
        return len(self.train_df)

In [3]:
with open(f'utils/output_classes.pkl', 'rb') as f:
    file = pickle.load(f)
all_classes = []
for class_group in file:
    all_classes.extend(class_group)
print(len(all_classes))

def expand_ec(ec):
    parts = ec.split('.')
    expanded = []
    for i in range(1, len(parts) + 1):
        expanded.append('.'.join(parts[:i]))
    return expanded

def hitec_dataset(row, train=True):
    sequence = row['Sequence']
    sequence = tokenizer.texts_to_sequences([sequence[:1023]])[0]
    sequence = [22] + sequence
    sequence += [0 for _ in range(1024-len(sequence))]
    
    if train == True:
        label = row['EC_List'].split(',')
    else:
        label = row['EC'].split(',')

    expanded_labels = set()
    
    for ec in label:
        ec = ec.strip()
        expanded_labels.update(expand_ec(ec))

    label = [1 if ec in expanded_labels else 0 for ec in all_classes]

    return sequence, label

2761


In [4]:
import os
import numpy as np
import pandas as pd
from skmultilearn.model_selection import IterativeStratification

def val_maker_one_fold_split(target_fold=1):
    base_path = f"data"
    
    train_df = pd.read_csv(f"{base_path}/dataset.csv")

    print("Extracting ECPICK features...")
    encoded_dataset_ec = []
    train_dataset = CustomDatasetECPICK(train_df=train_df)
    for idx, i in enumerate(train_dataset):
        if idx != 0 and idx % 50000 == 0: print(f"ECPICK: {idx}")
        encoded_dataset_ec.append(i)

    X_ec = np.array([sample[0] for sample in encoded_dataset_ec])
    y = np.array([sample[1] for sample in encoded_dataset_ec])

    print("Extracting HITEC features...")
    encoded_dataset_hi = []
    for idx, row in train_df.iterrows():
        if idx != 0 and idx % 50000 == 0: print(f"HITEC: {idx}")
        result = hitec_dataset(row)
        encoded_dataset_hi.append(result)
    X_hi = np.array([sample[0] for sample in encoded_dataset_hi])

    k_fold = IterativeStratification(n_splits=10, order=1)
    
    fold_chunks = []
    for _, chunk_indices in k_fold.split(X_ec, y):
        fold_chunks.append(chunk_indices)

    print(f"\nExtracting Single Fold (Fold {target_fold}) Split (Test:1, Val:1, Train:8)...\n")

    i = target_fold - 1 
    
    test_idx = fold_chunks[i]
    val_idx = fold_chunks[(i + 1) % 10]
    
    train_idx_list = []
    for j in range(10):
        if j != i and j != (i + 1) % 10:
            train_idx_list.append(fold_chunks[j])
    
    train_idx = np.concatenate(train_idx_list)

    x_train_ec, y_train = X_ec[train_idx], y[train_idx]
    x_val_ec, y_val = X_ec[val_idx], y[val_idx]
    x_test_ec, y_test = X_ec[test_idx], y[test_idx]
    
    x_train_hi = X_hi[train_idx]
    x_val_hi = X_hi[val_idx]
    x_test_hi = X_hi[test_idx]
    
    final_train_df = train_df.iloc[train_idx]
    final_val_df = train_df.iloc[val_idx]
    final_test_df = train_df.iloc[test_idx]

    train_list_ec = list(zip(x_train_ec, y_train))
    val_list_ec = list(zip(x_val_ec, y_val))
    test_list_ec = list(zip(x_test_ec, y_test))
    
    train_list_hi = list(zip(x_train_hi, y_train))
    val_list_hi = list(zip(x_val_hi, y_val))
    test_list_hi = list(zip(x_test_hi, y_test))

    os.makedirs(f"{base_path}/ECPICK/", exist_ok=True)
    os.makedirs(f"{base_path}/HITEC/", exist_ok=True)
    os.makedirs(f"{base_path}/CLEAN/", exist_ok=True)

    np.save(f'{base_path}/ECPICK/ecpick_train_fold{target_fold}.npy', np.array(train_list_ec, dtype=object))
    np.save(f'{base_path}/ECPICK/ecpick_val_fold{target_fold}.npy', np.array(val_list_ec, dtype=object))
    np.save(f'{base_path}/ECPICK/ecpick_test_fold{target_fold}.npy', np.array(test_list_ec, dtype=object))

    np.save(f'{base_path}/HITEC/hitec_train_fold{target_fold}.npy', np.array(train_list_hi, dtype=object))
    np.save(f'{base_path}/HITEC/hitec_val_fold{target_fold}.npy', np.array(val_list_hi, dtype=object))
    np.save(f'{base_path}/HITEC/hitec_test_fold{target_fold}.npy', np.array(test_list_hi, dtype=object))

    final_train_df.to_csv(f'{base_path}/CLEAN/train_clean_fold{target_fold}.csv', index=False)
    final_val_df.to_csv(f'{base_path}/CLEAN/val_clean_fold{target_fold}.csv', index=False)
    final_test_df.to_csv(f'{base_path}/CLEAN/test_clean_fold{target_fold}.csv', index=False)
    
    print(f"Fold {target_fold} Saved. (Train: {len(train_list_ec)}, Val: {len(val_list_ec)}, Test: {len(test_list_ec)})")
    print(f"\nSingle Fold ({target_fold}-Fold) split finished.\n")